# 🎮 Advanced NLP — Live Interactive Context-Switching Demonstrator
## Real-time Demonstration for TA & Professor Examination
**Team:** 5MinsBoss (Jai Aakash Gopalakrishnan, Aditya Nair, Chiranjeevi K, Rashmiraj Patra)  

---
### What this Demonstrator Does:
1. Allows the user/TA to choose a secret conversation topic $z$.
2. Demonstrates the arrival of outsider Agent C.
3. The trained **CoverController** observes the dialogue history and selects an optimal everyday cover topic $c$.
4. **Agent A** generates the metaphorical switched utterance in real time on a T4 GPU.
5. **Agent B** (insider) and **Agent C** (eavesdropper) simultaneously predict the secret topic, displaying live prediction probabilities.

In [ ]:
# Step 1: Initialize Environment and Model
import os, sys, torch
REPO_DIR = '/content/ANLP_grp_project'
if os.path.exists(REPO_DIR):
    sys.path.insert(0, REPO_DIR)
    %cd {REPO_DIR}

from proto_1.config import load_config, load_spec
from proto_1.lm import FrozenLM
from proto_1.pipeline import load_policy
from proto_1 import agents

cfg = load_config("proto_1/configs/colab_fast.json")
spec = load_spec("proto_1/data/spec.json")
lm = FrozenLM(cfg["model_name"], device=cfg["device"], dtype=cfg["dtype"])
controller = load_policy(cfg, spec, "rl_controller")
print("Interactive Demonstrator Ready! Model and Controller loaded.")

In [ ]:
# Step 2: Interactive Scenario Selector & Live Generation
topics = spec["hidden_topics"]
print("Available Secret Topics (z):")
for t in topics:
    print(f"  [{t['id']}] {t['label']}")

# Choose topic (change this ID to test any topic!)
SELECTED_Z = 0  # 0: Surprise Birthday Party, 1: Farewell Video, 2: Club Trivia, 3: Cafe Venue...

target_topic = topics[SELECTED_Z]
print("\n" + "=" * 75)
print(f"SELECTED SECRET TOPIC (z = {SELECTED_Z}): {target_topic['label']}")
print("=" * 75)

# Pre-switch history between A and B
sample_history = [
    ["A", "Are we still meeting up on Friday to finalize the arrangements?"],
    ["B", "Yes, definitely. We need to make sure everything stays quiet."],
    ["A", "Agreed, the timing has to be exact so nobody suspects anything."],
    ["B", "Right, let's keep the plan between us for now."]
]

print("\n--- PRE-SWITCH TRANSCRIPT (A and B chatting privately) ---")
for speaker, text in sample_history:
    print(f"{speaker}: {text}")

print("\n[🚨 OUTSIDER AGENT C JOINS THE CHAT ROOM 🚨]")

# 1. Controller observes state and selects cover topic
hist_feat = lm.features([agents.render_history(sample_history)], [cfg["features"]["layers"][-1]])[0, 0]
feat_tensor = torch.as_tensor(hist_feat, dtype=torch.float32).unsqueeze(0)
z_tensor = torch.tensor([SELECTED_Z], dtype=torch.long)
with torch.no_grad():
    action_logits = controller(feat_tensor, z_tensor)
    chosen_action = action_logits.argmax(-1).item()

cover_topic = spec["cover_topics"][chosen_action]
print(f"\n🧠 Controller Decision: Switch conversation to everyday cover topic: -> '{cover_topic}'")

# 2. Agent A generates switched turn
system_prompt, user_prompt, prefix = agents.a_prompt(
    spec, "rl_controller", sample_history, {"z": SELECTED_Z, "private_desc": target_topic["private_desc"].format(person="Alex", day="Friday", club="Film")},
    chosen_action, "demo_example"
)
full_prompt = lm.chat(system_prompt, user_prompt)
raw_output = lm.generate([full_prompt], max_new_tokens=60, temperature=0.7, top_p=0.9, do_sample=True, batch_size=1)[0]
switched_message = agents.clean_message(prefix, raw_output)

print(f"\n🗣️ Agent A's Switched Utterance (Public):\n\" {switched_message} \"")

# 3. Eavesdropper C guesses z
c_sys, c_usr = agents.c_prompt(spec, switched_message)
c_raw = lm.generate([lm.chat(c_sys, c_usr)], max_new_tokens=4, do_sample=False, batch_size=1)[0]
c_pred = agents.parse_label(c_raw, len(topics))

# 4. Informed Insider B guesses z
b_sys, b_usr = agents.b_prompt(spec, sample_history, switched_message, "informed")
b_raw = lm.generate([lm.chat(b_sys, b_usr)], max_new_tokens=4, do_sample=False, batch_size=1)[0]
b_pred = agents.parse_label(b_raw, len(topics))

print("\n" + "=" * 75)
print("LIVE EVALUATION RESULTS:")
print(f"  • Insider B Guess   : [{b_pred}] {topics[b_pred]['label'] if b_pred >= 0 else 'Invalid'} -> {'SUCCESS (B decoded z!)' if b_pred == SELECTED_Z else 'FAILED'}")
print(f"  • Outsider C Guess  : [{c_pred}] {topics[c_pred]['label'] if c_pred >= 0 else 'Invalid'} -> {'BLINDED (C fooled!)' if c_pred != SELECTED_Z else 'DETECTED'}")
print(f"  • Joint Coordination: {'PASSED (B Correct & C Blinded)' if (b_pred == SELECTED_Z and c_pred != SELECTED_Z) else 'FAILED'}")
print("=" * 75)